# Keyword analysis

In [1]:
# import sys
#
# if ".." not in sys.path:
#     sys.path.append("..")

import html

import great_tables as gt
import numpy as np
import polars as pl
from scipy.stats import chi2_contingency, fisher_exact

import polars_corpus as plc

In [37]:
bnc = pl.scan_parquet("bnc.parquet").set_sorted("file_id")
speakers = pl.scan_parquet("bnc-speakers.parquet")

speakers = speakers.filter(pl.col("sex").is_in(["m", "f"]))
speakers.group_by("sex").len()

bnc = (
    bnc.filter(pl.col("text_type") == "CONVRSN")
    .join(speakers, on="speaker_id", how="inner")
    .with_columns(pl.col("token").str.to_lowercase().alias("norm"))
)

In [3]:
table = (
    bnc.group_by((pl.col("norm") == "husband").alias("is_husband"), "sex")
    .agg(pl.col("norm").len().alias("count"))
    .with_columns(rel_freq=pl.col("count") / pl.col("count").sum().over("sex"))
    .sort(by=["sex", "is_husband"])
).collect()
table

is_husband,sex,count,rel_freq
bool,str,u32,f64
false,"""f""",2745317,0.999918
true,"""f""",225,0.000082
false,"""m""",1784664,0.999967
true,"""m""",59,0.000033


In [4]:
table = np.array(table["count"]).reshape(2, 2)
table

array([[2745317,     225],
       [1784664,      59]], dtype=uint32)

In [5]:
fisher_exact(table)

SignificanceResult(statistic=np.float64(0.4033717968449212), pvalue=np.float64(3.008076355152558e-11))

In [6]:
chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(40.47101710487801), pvalue=np.float64(1.9955429957285215e-10), dof=1, expected_freq=array([[2.74536988e+06, 1.72116626e+02],
       [1.78461112e+06, 1.11883374e+02]]))

In [7]:
np.sqrt(chi2_contingency(table).statistic / table.sum())

np.float64(0.0029888922299691526)

In [8]:
table = (
    bnc.group_by((pl.col("norm") == "wife").alias("is_wife"), "sex")
    .agg(pl.col("norm").len().alias("count"))
    .with_columns(rel_freq=pl.col("count") / pl.col("count").sum().over("sex"))
    .sort(by=["sex", "is_wife"])
).collect()
table = np.array(table["count"]).reshape(2, 2)
table

array([[2745393,     149],
       [1784587,     136]], dtype=uint32)

In [9]:
chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(7.926018721265425), pvalue=np.float64(0.004872890089612014), dof=1, expected_freq=array([[2.74536928e+06, 1.72722671e+02],
       [1.78461072e+06, 1.12277329e+02]]))

In [10]:
np.sqrt(chi2_contingency(table).statistic / table.sum())

np.float64(0.0013227133699140585)

In [11]:
np.sqrt(chi2_contingency(table).statistic / table.sum())

np.float64(0.0013227133699140585)

In [12]:
fisher_exact(table)

SignificanceResult(statistic=np.float64(1.404169181504793), pvalue=np.float64(0.004360351139967793))

In [13]:
table = (
    bnc.group_by((pl.col("norm") == "is").alias("is_is"), "sex")
    .agg(pl.col("norm").len().alias("count"))
    .with_columns(rel_freq=pl.col("count") / pl.col("count").sum().over("sex"))
    .sort(by=["sex", "is_is"])
).collect()
table = np.array(table["count"]).reshape(2, 2)
table

array([[2728205,   17337],
       [1771446,   13277]], dtype=uint32)

In [14]:
chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(203.65773353907946), pvalue=np.float64(3.3240516542754505e-46), dof=1, expected_freq=array([[2726988.55493928,   18553.44506072],
       [1772662.44506072,   12060.55493928]]))

In [15]:
np.sqrt(chi2_contingency(table).statistic / table.sum())

np.float64(0.006704843567080365)

In [16]:
fisher_exact(table)

SignificanceResult(statistic=np.float64(1.1794379252713856), pvalue=np.float64(8.788243292335871e-46))

-----

Stefanowitsch (2020), pp. 378–380

In [40]:
male_keywords = plc.keywords(
    target = bnc.filter(pl.col('sex')=='m'),
    reference= bnc.filter(pl.col('sex')=='f'),
    method='ll',
    expr=pl.col('token').str.to_lowercase()
).collect(engine="streaming")

In [41]:
female_keywords = plc.keywords(
    target = bnc.filter(pl.col('sex')=='f'),
    reference= bnc.filter(pl.col('sex')=='m'),
    method='ll',
    expr=pl.col('token').str.to_lowercase()
).collect(engine="streaming")

In [39]:
tbl = (
    male_keywords.head(20)
    .select("token",
            pl.col('freqs').struct.field('f12').alias('male_freq'),
            (pl.col('freqs').struct.field('f1')-pl.col('freqs').struct.field('f12')).alias('female_freq'),
            'LogLik')
    .style.fmt_number(["LogLik"], decimals=2)
    .fmt_integer(["male_freq", "female_freq"], use_seps=True)
    .fmt(html.escape, columns=["token"])
    .cols_label(
        {
            "token": "word",
            "LogLik": "LL",
            "male_freq": "m freq",
            "female_freq": "f freq",
        }
    )
    .opt_row_striping()
    .opt_vertical_padding(0.6)
)

#tbl.save("LL")
tbl

word,m freq,f freq,LL
fucking,"1,383",326,"1,237.92"
er,"9,415","9,337",900.79
the,"43,385","57,367",574.38
",","84,406","119,150",380.47
yeah,"21,888","28,793",305.65
minus,257,35,299.75
<unclear/>,"30,659","41,710",269.42
aye,"1,164",876,258.85
hundred,"1,473","1,233",249.38
right,"6,081","7,092",249.24


In [40]:
tbl = (
    female_keywords.head(20)
    .select("token",
            pl.col('freqs').struct.field('f12').alias('male_freq'),
            (pl.col('freqs').struct.field('f1')-pl.col('freqs').struct.field('f12')).alias('female_freq'),
            'LogLik')
    .style.fmt_number(["LogLik"], decimals=2)
    .fmt_integer(["male_freq", "female_freq"], use_seps=True)
    .fmt(html.escape, columns=["token"])
    .cols_label(
        {
            "token": "word",
            "LogLik": "LL",
            "male_freq": "m freq",
            "female_freq": "f freq",
        }
    )
    .opt_row_striping()
    .opt_vertical_padding(0.6)
)

#tbl.save("LL")
tbl

word,m freq,f freq,LL
she,"22,807","7,037","3,373.99"
her,"7,306","2,313","1,017.04"
said,"12,375","4,911",915.45
n't,"44,380","24,221",494.18
i,"93,330","54,825",369.24
and,"50,467","29,109",271.30
to,"40,934","23,693",207.09
cos,"6,864","3,314",204.47
christmas,"1,005",285,175.17
oh,"23,472","13,236",174.57


Lijffijt et al. (2016)

In [42]:
male_keywords = plc.keywords(
    target = bnc.filter(pl.col('sex')=='m'),
    reference= bnc.filter(pl.col('sex')=='f'),
    method='ttest',
    expr=pl.col('token').str.to_lowercase()
).collect(engine="streaming")

In [43]:
female_keywords = plc.keywords(
    target = bnc.filter(pl.col('sex')=='f').lazy(),
    reference= bnc.filter(pl.col('sex')=='m'),
    method='ttest',
    expr=pl.col('token').str.to_lowercase()
).collect(engine="streaming")